# 16 优化算法详解 - SGD、Momentum、Adam

在上一篇教程中，我们学习了梯度下降的基本原理。本教程将深入讲解深度学习中常用的**优化算法**，它们决定了模型参数如何更新，直接影响训练的速度和质量。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、回顾：梯度下降的三种形式

### 1. 批量梯度下降（Batch Gradient Descent）
每次使用**全部训练数据**计算梯度：
$$\theta := \theta - \eta \cdot \frac{1}{N}\sum_{i=1}^{N}\nabla_\theta L(f(x^{(i)}; \theta), y^{(i)})$$

- 优点：梯度稳定，收敛到最优解
- 缺点：数据量大时计算慢，内存消耗大

### 2. 随机梯度下降（SGD）
每次只用**一个样本**更新：
$$\theta := \theta - \eta \cdot \nabla_\theta L(f(x^{(i)}; \theta), y^{(i)})$$

- 优点：计算快，可以在线学习
- 缺点：梯度波动大，收敛不稳定

### 3. 小批量梯度下降（Mini-Batch GD）
每次使用**一小批数据**（通常32-512个）：
$$\theta := \theta - \eta \cdot \frac{1}{B}\sum_{i=1}^{B}\nabla_\theta L(f(x^{(i)}; \theta), y^{(i)})$$

- 兼顾稳定性和效率，**最常用**

In [ ]:
# 对比三种梯度下降的行为
def rastrigin(x, y):
    """Rastrigin函数：有很多局部最小值，适合测试优化器"""
    return x**2 + y**2 - 0.5 * torch.cos(3*x) - 0.5 * torch.cos(3*y) + 1

def grad_rastrigin(x, y):
    """Rastrigin函数的梯度"""
    dx = 2*x + 1.5 * torch.sin(3*x)
    dy = 2*y + 1.5 * torch.sin(3*y)
    return dx, dy

# 生成损失函数曲面
x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = X**2 + Y**2 - 0.5*np.cos(3*X) - 0.5*np.cos(3*Y) + 1

fig, ax = plt.subplots(figsize=(8, 7))
contour = ax.contourf(X, Y, Z, levels=40, cmap='viridis')
ax.plot([0], [0], 'r*', markersize=15, label='全局最小值 (0, 0)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Rastrigin函数等高线\n(多局部最小值的测试函数)')
ax.legend()
fig.colorbar(contour, ax=ax, label='Loss')
plt.tight_layout()
plt.show()

# 模拟SGD、Batch GD、Mini-Batch GD
torch.manual_seed(42)

def simulate_gd(mode, lr=0.05, epochs=100):
    """
    mode: 'batch', 'sgd', 'minibatch'
    """
    x, y = 1.8, 1.5  # 初始位置
    path_x, path_y, path_loss = [x], [y], [rastrigin(torch.tensor(x), torch.tensor(y)).item()]
    
    for epoch in range(epochs):
        if mode == 'batch':
            # 精确梯度（无噪声）
            dx, dy = grad_rastrigin(torch.tensor(x), torch.tensor(y))
        elif mode == 'sgd':
            # 带噪声的梯度
            dx, dy = grad_rastrigin(torch.tensor(x), torch.tensor(y))
            dx += 0.3 * torch.randn(1)
            dy += 0.3 * torch.randn(1)
        else:  # minibatch
            # 中等噪声
            dx, dy = grad_rastrigin(torch.tensor(x), torch.tensor(y))
            dx += 0.1 * torch.randn(1)
            dy += 0.1 * torch.randn(1)
        
        x = x - lr * dx.item()
        y = y - lr * dy.item()
        x = np.clip(x, -2.5, 2.5)
        y = np.clip(y, -2.5, 2.5)
        path_x.append(x)
        path_y.append(y)
        path_loss.append(rastrigin(torch.tensor(x), torch.tensor(y)).item())
    
    return path_x, path_y, path_loss

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
modes = [('batch', '批量GD', 'blue'), ('sgd', '随机GD', 'red'), ('minibatch', '小批量GD', 'green')]

for idx, (mode, label, color) in enumerate(modes):
    ax = axes[idx]
    ax.contourf(X, Y, Z, levels=40, cmap='viridis', alpha=0.8)
    
    px, py, pl = simulate_gd(mode)
    ax.plot(px, py, 'o-', color=color, linewidth=1.5, markersize=3, alpha=0.7, label=label)
    ax.plot([0], [0], 'r*', markersize=12)
    ax.set_title(f'{label}\n最终位置: ({px[-1]:.3f}, {py[-1]:.3f})\n最终损失: {pl[-1]:.4f}')
    ax.legend()

plt.tight_layout()
plt.show()

## 二、动量法（Momentum）

### 问题：SGD的震荡

标准SGD在每个步骤只根据当前梯度更新，不考虑之前的方向。这导致：
- 在狭窄的山谷状损失函数中会**来回震荡**
- 收敛速度很慢

### 解决方案：引入动量

动量法模拟物理中的动量概念，维护一个**速度向量**，它累积过去的梯度方向：

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot \nabla_\theta L(\theta)$$
$$\theta := \theta - \eta \cdot v_t$$

其中：
- $v_t$ 是速度（velocity）
- $\beta$ 是动量系数（通常取0.9）
- 过去的梯度通过指数衰减影响当前速度

### 直观理解

想象一个球从山上滚下：
- 它会**积累速度**，越来越快
- 如果遇到小坑（局部最小值），由于惯性它会**冲过去**
- 在梯度方向一致时加速，在梯度方向变化时减速

In [ ]:
# 对比SGD和Momentum在"山谷"函数上的表现
def valley_loss(x, y):
    """狭长的山谷状损失函数"""
    return 0.1 * x**2 + 2.0 * y**2

def grad_valley(x, y):
    return 0.2 * x, 4.0 * y

# SGD
def run_sgd(x0, y0, lr, epochs):
    x, y = x0, y0
    path = [(x, y)]
    for _ in range(epochs):
        dx, dy = grad_valley(x, y)
        x -= lr * dx
        y -= lr * dy
        path.append((x, y))
    return path

# Momentum
def run_momentum(x0, y0, lr, beta, epochs):
    x, y = x0, y0
    vx, vy = 0.0, 0.0
    path = [(x, y)]
    for _ in range(epochs):
        dx, dy = grad_valley(x, y)
        vx = beta * vx + (1 - beta) * dx
        vy = beta * vy + (1 - beta) * dy
        x -= lr * vx
        y -= lr * vy
        path.append((x, y))
    return path

# 生成等高线
x_r = np.linspace(-3, 3, 100)
y_r = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_r, y_r)
Z = valley_loss(X, Y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SGD
path_sgd = run_sgd(2.5, 2.5, lr=0.15, epochs=50)
path_sgd = np.array(path_sgd)
ax = axes[0]
ax.contourf(X, Y, Z, levels=30, cmap='coolwarm')
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'ko-', linewidth=1.5, markersize=3, label='SGD')
ax.plot([0], [0], 'r*', markersize=12, label='最小值')
ax.set_title(f'SGD (lr=0.15, 50步)\n最终: ({path_sgd[-1,0]:.3f}, {path_sgd[-1,1]:.3f})')
ax.legend()

# Momentum
path_mom = run_momentum(2.5, 2.5, lr=0.5, beta=0.9, epochs=50)
path_mom = np.array(path_mom)
ax = axes[1]
ax.contourf(X, Y, Z, levels=30, cmap='coolwarm')
ax.plot(path_mom[:, 0], path_mom[:, 1], 'go-', linewidth=1.5, markersize=3, label='Momentum')
ax.plot([0], [0], 'r*', markersize=12, label='最小值')
ax.set_title(f'Momentum (lr=0.5, β=0.9, 50步)\n最终: ({path_mom[-1,0]:.3f}, {path_mom[-1,1]:.3f})')
ax.legend()

plt.tight_layout()
plt.show()

# 打印详细对比
print("SGD vs Momentum 详细对比（山谷函数）")
print(f"{'步数':<6} {'SGD x':>10} {'SGD y':>10} {'SGD Loss':>12} || {'Mom x':>10} {'Mom y':>10} {'Mom Loss':>12}")
print("-" * 90)
for i in [0, 5, 10, 20, 30, 40, 50]:
    if i < len(path_sgd) and i < len(path_mom):
        sgd_loss = valley_loss(path_sgd[i,0], path_sgd[i,1])
        mom_loss = valley_loss(path_mom[i,0], path_mom[i,1])
        print(f"{i:<6} {path_sgd[i,0]:>10.4f} {path_sgd[i,1]:>10.4f} {sgd_loss:>12.4f} || {path_mom[i,0]:>10.4f} {path_mom[i,1]:>10.4f} {mom_loss:>12.4f}")

## 三、自适应学习率方法

### 问题：统一学习率的局限

SGD和Momentum对所有参数使用相同的学习率。但实际情况是：
- **频繁更新的参数**应该用小学习率（稳定）
- **稀疏更新的参数**应该用大学习率（加速）

### AdaGrad：累积梯度平方

$$G_t = G_{t-1} + (\nabla_\theta L)^2$$
$$\theta := \theta - \frac{\eta}{\sqrt{G_t + \epsilon}} \odot \nabla_\theta L$$

缺点：$G_t$ 单调递增，学习率会越来越小，最终停止学习。

### RMSProp：指数衰减的梯度平方平均

RMSProp改进了AdaGrad，用指数衰减代替累加：

$$E[g^2]_t = \gamma \cdot E[g^2]_{t-1} + (1 - \gamma) \cdot (\nabla_\theta L)^2$$
$$\theta := \theta - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \odot \nabla_\theta L$$

其中 $\gamma$ 通常取 0.9。

In [ ]:
# 实现RMSProp并可视化
def run_rmsprop(x0, y0, lr, gamma, epochs):
    x, y = x0, y0
    eg2_x, eg2_y = 0.0, 0.0  # E[g^2]的估计
    path = [(x, y)]
    
    for _ in range(epochs):
        dx, dy = grad_valley(x, y)
        
        # 更新梯度平方平均
        eg2_x = gamma * eg2_x + (1 - gamma) * dx**2
        eg2_y = gamma * eg2_y + (1 - gamma) * dy**2
        
        # 自适应学习率更新
        x -= lr * dx / (np.sqrt(eg2_x) + 1e-8)
        y -= lr * dy / (np.sqrt(eg2_y) + 1e-8)
        path.append((x, y))
    
    return np.array(path)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

optimizers = [
    ('SGD', lambda: run_sgd(2.5, 2.5, lr=0.15, epochs=50), 'blue'),
    ('Momentum', lambda: run_momentum(2.5, 2.5, lr=0.5, beta=0.9, epochs=50), 'green'),
    ('RMSProp', lambda: run_rmsprop(2.5, 2.5, lr=0.1, gamma=0.9, epochs=50), 'orange'),
]

for idx, (name, run_fn, color) in enumerate(optimizers):
    ax = axes[idx]
    ax.contourf(X, Y, Z, levels=30, cmap='coolwarm')
    path = run_fn()
    ax.plot(path[:, 0], path[:, 1], color=color, linewidth=1.5, markersize=4, marker='o', alpha=0.7)
    ax.plot([0], [0], 'r*', markersize=12)
    ax.set_title(f'{name}\n最终: ({path[-1,0]:.4f}, {path[-1,1]:.4f})\nLoss: {valley_loss(path[-1,0], path[-1,1]):.6f}')

plt.tight_layout()
plt.show()

## 四、Adam优化器（核心重点）

### Adam = Momentum + RMSProp

Adam（Adaptive Moment Estimation）结合了Momentum和RMSProp的优点，是深度学习中最流行的优化器。

### Adam的四个步骤

#### 步骤1：计算梯度的一阶矩（动量）
$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

#### 步骤2：计算梯度的二阶矩（自适应学习率）
$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

#### 步骤3：偏差修正（因为初始值为0，早期估计有偏差）
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

#### 步骤4：更新参数
$$\theta := \theta - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

### 默认超参数
- $\eta = 0.001$（学习率）
- $\beta_1 = 0.9$（一阶矩衰减）
- $\beta_2 = 0.999$（二阶矩衰减）
- $\epsilon = 10^{-8}$（数值稳定性）

In [ ]:
# 从零实现Adam优化器
def run_adam(x0, y0, lr, beta1, beta2, eps, epochs):
    x, y = x0, y0
    m_x, m_y = 0.0, 0.0  # 一阶矩
    v_x, v_y = 0.0, 0.0  # 二阶矩
    path = [(x, y)]
    
    for t in range(1, epochs + 1):
        dx, dy = grad_valley(x, y)
        
        # 步骤1: 一阶矩（动量）
        m_x = beta1 * m_x + (1 - beta1) * dx
        m_y = beta1 * m_y + (1 - beta1) * dy
        
        # 步骤2: 二阶矩（自适应学习率）
        v_x = beta2 * v_x + (1 - beta2) * dx**2
        v_y = beta2 * v_y + (1 - beta2) * dy**2
        
        # 步骤3: 偏差修正
        m_hat_x = m_x / (1 - beta1**t)
        m_hat_y = m_y / (1 - beta1**t)
        v_hat_x = v_x / (1 - beta2**t)
        v_hat_y = v_y / (1 - beta2**t)
        
        # 步骤4: 更新参数
        x -= lr * m_hat_x / (np.sqrt(v_hat_x) + eps)
        y -= lr * m_hat_y / (np.sqrt(v_hat_y) + eps)
        path.append((x, y))
    
    return np.array(path)

# Adam详细数值演示
print("=" * 70)
print("Adam优化器 详细数值演示")
print("=" * 70)

x, y = 2.0, 1.5
lr, beta1, beta2, eps = 0.1, 0.9, 0.999, 1e-8
m_x, m_y = 0.0, 0.0
v_x, v_y = 0.0, 0.0

print(f"初始: x={x:.4f}, y={y:.4f}, Loss={valley_loss(x,y):.6f}")
print(f"超参数: lr={lr}, β1={beta1}, β2={beta2}, ε={eps}")
print()

for t in range(1, 8):
    dx, dy = grad_valley(x, y)
    
    # Adam步骤
    m_x = beta1 * m_x + (1 - beta1) * dx
    m_y = beta1 * m_y + (1 - beta1) * dy
    v_x = beta2 * v_x + (1 - beta2) * dx**2
    v_y = beta2 * v_y + (1 - beta2) * dy**2
    
    m_hat_x = m_x / (1 - beta1**t)
    m_hat_y = m_y / (1 - beta1**t)
    v_hat_x = v_x / (1 - beta2**t)
    v_hat_y = v_y / (1 - beta2**t)
    
    x -= lr * m_hat_x / (np.sqrt(v_hat_x) + eps)
    y -= lr * m_hat_y / (np.sqrt(v_hat_y) + eps)
    
    loss = valley_loss(x, y)
    print(f"Step {t}: x={x:.6f}, y={y:.6f}, Loss={loss:.6f} (dx={dx:.4f}, dy={dy:.4f})")

# 对比所有优化器
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

all_opts = [
    ('SGD', lambda: run_sgd(2.5, 2.5, lr=0.15, epochs=50), 'blue'),
    ('Momentum', lambda: run_momentum(2.5, 2.5, lr=0.5, beta=0.9, epochs=50), 'green'),
    ('RMSProp', lambda: run_rmsprop(2.5, 2.5, lr=0.1, gamma=0.9, epochs=50), 'orange'),
    ('Adam', lambda: run_adam(2.5, 2.5, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, epochs=50), 'red'),
]

for idx, (name, run_fn, color) in enumerate(all_opts):
    ax = axes[idx]
    ax.contourf(X, Y, Z, levels=30, cmap='coolwarm')
    path = run_fn()
    ax.plot(path[:, 0], path[:, 1], color=color, linewidth=1.5, markersize=4, marker='o', alpha=0.7)
    ax.plot([0], [0], 'r*', markersize=12)
    ax.set_title(f'{name}\n最终: ({path[-1,0]:.4f}, {path[-1,1]:.4f})\nLoss: {valley_loss(path[-1,0], path[-1,1]):.6f}')

plt.tight_layout()
plt.show()

## 五、在真实神经网络中对比优化器

让我们在真实的神经网络训练任务中对比各种优化器。

In [ ]:
# 在真实神经网络中对比优化器
torch.manual_seed(42)

# 数据：拟合 y = x^2
X = torch.linspace(-2, 2, 100).unsqueeze(1)
y_true = X**2
y = y_true + 0.3 * torch.randn(100, 1)

class SmallNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

optimizers_config = [
    ('SGD', lambda params: optim.SGD(params, lr=0.05), 'blue'),
    ('SGD+Momentum', lambda params: optim.SGD(params, lr=0.05, momentum=0.9), 'green'),
    ('Adam', lambda params: optim.Adam(params, lr=0.01), 'red'),
    ('RMSprop', lambda params: optim.RMSprop(params, lr=0.01), 'orange'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, opt_fn, color) in enumerate(optimizers_config):
    torch.manual_seed(42)
    model = SmallNN()
    criterion = nn.MSELoss()
    optimizer = opt_fn(model.parameters())
    
    loss_hist = []
    for epoch in range(200):
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
    
    ax = axes[idx]
    ax.plot(loss_hist, color=color, linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'{name}\n最终损失: {loss_hist[-1]:.6f}')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # 显示拟合结果
    model.eval()
    with torch.no_grad():
        y_pred = model(X)

print(f"{'Optimizer':<15} {'最终MSE':>12}")
print("-" * 30)
for name, opt_fn, color in optimizers_config:
    torch.manual_seed(42)
    model = SmallNN()
    optimizer = opt_fn(model.parameters())
    criterion = nn.MSELoss()
    for epoch in range(200):
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"{name:<15} {loss.item():>12.6f}")

plt.tight_layout()
plt.show()

## 六、学习率调度器（Learning Rate Scheduler）

除了选择合适的优化器，**动态调整学习率**也是训练的关键策略。

### 常见调度策略

1. **StepLR**：每N步衰减
   $$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / \text{step} \rfloor}$$

2. **CosineAnnealing**：余弦衰减到最小值
   $$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_0 - \eta_{\min})(1 + \cos(\frac{t}{T}\pi))$$

3. **ReduceLROnPlateau**：损失不下降时衰减

In [ ]:
# 学习率调度器对比
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau

schedulers_config = [
    ('固定学习率', None, 'blue'),
    ('StepLR (每50步×0.5)', 'step', 'green'),
    ('CosineAnnealing', 'cosine', 'orange'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, (name, sched_type, color) in enumerate(schedulers_config):
    torch.manual_seed(42)
    model = SmallNN()
    optimizer = optim.Adam(model.parameters(), lr=0.05)
    criterion = nn.MSELoss()
    
    if sched_type == 'step':
        scheduler = StepLR(optimizer, step_size=50, gamma=0.5)
    elif sched_type == 'cosine':
        scheduler = CosineAnnealingLR(optimizer, T_max=200, eta_min=0.001)
    else:
        scheduler = None
    
    loss_hist = []
    lr_hist = []
    
    for epoch in range(200):
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
        lr_hist.append(optimizer.param_groups[0]['lr'])
        
        if scheduler is not None:
            scheduler.step()
    
    ax = axes[idx]
    ax.plot(loss_hist, color=color, linewidth=2, label='Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss', color=color)
    ax.set_title(f'{name}\n最终损失: {loss_hist[-1]:.6f}')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    ax2 = ax.twinx()
    ax2.plot(lr_hist, 'gray', linewidth=1, linestyle='--', label='LR')
    ax2.set_ylabel('Learning Rate', color='gray')

plt.tight_layout()
plt.show()

# 打印各调度器的最终损失
print(f"{'调度器':<30} {'最终MSE':>12}")
print("-" * 45)
for name, sched_type, color in schedulers_config:
    torch.manual_seed(42)
    model = SmallNN()
    optimizer = optim.Adam(model.parameters(), lr=0.05)
    if sched_type == 'step':
        scheduler = StepLR(optimizer, step_size=50, gamma=0.5)
    elif sched_type == 'cosine':
        scheduler = CosineAnnealingLR(optimizer, T_max=200, eta_min=0.001)
    else:
        scheduler = None
    criterion = nn.MSELoss()
    for epoch in range(200):
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler: scheduler.step()
    print(f"{name:<30} {loss.item():>12.6f}")

## 七、本章总结

### 优化器对比表

| 优化器 | 核心思想 | 优点 | 缺点 | 适用场景 |
|--------|---------|------|------|----------|
| **SGD** | 沿梯度反方向更新 | 简单，理论清晰 | 收敛慢，易震荡 | 简单任务 |
| **SGD+Momentum** | 引入速度累积历史梯度 | 加速收敛，减少震荡 | 需要调β | 大多数任务 |
| **RMSProp** | 用梯度平方自适应学习率 | 参数自适应 | 没有动量 | RNN训练 |
| **Adam** | Momentum + RMSProp | 收敛快，自适应 | 可能过拟合 | **首选默认** |

### 实用建议

1. **默认使用 Adam**：大多数情况下Adam效果最好
2. **精细调优时用 SGD+Momentum**：可能获得更好的泛化性能
3. **配合学习率调度器**：CosineAnnealing是很好的选择
4. **学习率建议**：Adam用 0.001~0.01，SGD用 0.01~0.1

### 下一篇预告

下一篇将深入讲解**反向传播（Backpropagation）**的完整细节：
- 链式法则的数学推导
- 手动计算一个完整神经网络的反向传播
- 梯度消失和梯度爆炸问题